# 11 — Measurement source and timestamp audit

**What do our exported Garmin clocks and IBI rows establish?**
This notebook explains existing evidence and missing provenance. It does not fit a model, calculate
new real HRV, reread sensor/EMA rows, or change a policy. All 32 final-test participants stay sealed.
The collection SDK/API and version remain unidentified. A proposed device/SDK integration is not
a collection-time contract for these exports.

Read the [specification](../reports/measurement_source_specification.md). Public tables contain
aggregate/schema information only; discovery paths and study-document extracts remain private.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
from garmin_preprocessing.common import verify_bundle  # noqa: E402
from garmin_inventory.privacy import sha256  # noqa: E402
from garmin_measurement_source.audit import synthetic_ambiguity  # noqa: E402

BUNDLE = ROOT / "private/measurement_source/runs/96d7800ef13ab00b"
REPORT = ROOT / "reports/measurement_source/1c44cb036f4ba779"
manifest = verify_bundle(BUNDLE)
report_manifest = verify_bundle(REPORT)
assert manifest["final_test_access"] is False and manifest["no_new_calculations"]
config = json.loads((ROOT / "config/measurement_source.json").read_text())
for source, expected in config["reviewed_sha256"].items():
    assert sha256(ROOT / source) == expected
assert sha256(BUNDLE / "manifest.json") == report_manifest["inputs"]["private_evidence_manifest_sha256"]
audit = json.loads((REPORT / "audit_summary.json").read_text())
print({key: audit[key] for key in ["collection_sdk_and_version", "application_source_candidates", "application_build_candidates", "new_participant_records_read", "model_fits", "real_hrv_calculations"]})


## What has been tested

Notebooks 04–10 test **HR/activity prediction** of the continuous three-item composite. Notebook 03
only calculated five IBI variability measures in 20 selected run-hypothesis rows. It tested numerical
feasibility, not IBI–stress prediction or physiological validity.

On the matching 60-minute HR cohort (1,634 responses/118 people), HGB context+HR MAE was 2.0814,
matching median 2.0710 and HGB context 2.0741. The HR−median difference was +0.0104, conditional
interval [−0.0160, 0.0392]. Supported steps and earlier-reference deviations also did not establish
improvement within their separate matched cohorts. Do not compare unpaired errors across cohorts.
See the report for the exact cohort-by-cohort contrasts and source links.

The 53.5% within-person score variation is a descriptive decomposition, **not the fraction HR can
predict**. Scores can change for reasons absent from the wearable or through reporting noise.
Full-record means use future information and stay out of predictors. In the native HR cohort,
the person-centered mean-HR association is small and its uncertainty includes zero.

## Where the evidence chain stops

Device → SDK/API payload → callback/storage → CSV exporter: **missing implementation evidence**.

CSV strings → preserved physical records → logical view → conditional windows: **traceable locally**.

`R61-R33_ResearchStrategy.pdf` p.5 describes a Vivoactive study; p.7 proposes a Vivoactive 5/SDK
integration. It supplies neither the edition/version nor the clock assignment used by our exports.
Official Garmin documentation describes several possible sources and processing modes; none is
identified by scalar `val` and fractional `ts` alone.

In [ ]:
display(pd.DataFrame(audit["evidence"])[["key", "path", "line", "conclusion"]])
display(pd.DataFrame(json.loads((REPORT / "official_sources.json").read_text())))
print(audit["raw_schema"])


## A synthetic ambiguity: the same CSV can have two meanings

Two identical scalar rows can mean a retried event or two equal intervals flattened from one batch.
Neither row shape nor retention identifies which happened. Source event/batch identity and the
flattening code are needed. This example does not reconstruct any beat timestamps.

In [ ]:
example = synthetic_ambiguity()
assert not example["distinguishable_from_export_alone"]
display(pd.DataFrame(example["scalar_rows"]))  # Synthetic rows only.
print(example["history_a"])
print(example["history_b"])
print("Physical rows survive; logical exact-duplicate collapse alone cannot settle event identity.")


## Reused timing and continuity evidence

Immediate five-minute windows: 2,196 contain no IBI, 1,510 contain some; Screen admits four end/five
start windows. Short runs, gaps and support losses overlap. Removing the provisional >20% change
gate alone did not restore retention. Fractional precision, compressed pairs and gaps are compatible
with batching or other clock/export behavior, but do not diagnose acquisition semantics.

Both anchoring hypotheses can be wrong under receipt clocks. A Screen pass can contain only a
120-second uninterrupted run; the ≥240-second pilot is still not complete five-minute HRV. Package
agreement establishes arithmetic only. No rule changes or new HRV calculations follow here.
Historical 12/24-hour audits were superseded in Notebook 04; only unaffected earlier evidence is reused.

In [ ]:
display(pd.read_json(REPORT / "rule_dependencies.json"))
display(pd.read_json(REPORT / "feature_status.json"))


## Provider knowledge versus testable implementation choices

Provider evidence must identify clocks, SDK/API release, array order/anchors, device/firmware,
quality flags, missing beats and prompt events. Once supplied, we can independently test the
payload-to-row mapping, preserve interval identity and version any justified correction.
We cannot test an undocumented clock into certainty by selecting a threshold that retains more rows.

The next step is a tiny redacted provenance trace, not another model search. No provider message
has been sent. If original event clocks/order were irretrievably lost, expanded physiological HRV
and exact recall-aligned timing remain blocked.

In [ ]:
display(pd.read_json(REPORT / "decision_table.json"))
display(Markdown((REPORT / "provider_request_draft.md").read_text()))
print("Report:", REPORT.relative_to(ROOT) / "report.md")
print("Professor briefing:", REPORT.relative_to(ROOT) / "professor_provider_briefing.md")
print("Ignored evidence bundle:", BUNDLE.relative_to(ROOT))
